# 🚀 DocStrange GPU Mode - Google Colab

Questo notebook ti permette di utilizzare DocStrange con GPU **gratuita** su Google Colab.

## Prima di iniziare:
1. **Attiva la GPU**: `Runtime` → `Change runtime type` → `Hardware accelerator` → **GPU**
2. Esegui le celle in ordine

---

## 1️⃣ Verifica GPU Disponibile

In [ ]:
# Verifica che la GPU sia attiva
!nvidia-smi

import torch
print(f"\n✅ CUDA disponibile: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 Memoria GPU: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️ GPU non disponibile! Vai su Runtime > Change runtime type > GPU")

## 2️⃣ Installazione DocStrange

In [ ]:
# Installa DocStrange con tutte le dipendenze
!pip install -q git+https://github.com/NanoNets/nanonets.ocr-2.git

# Installa dipendenze aggiuntive per GPU mode
!apt-get install -qq poppler-utils tesseract-ocr

print("✅ Installazione completata!")

## 3️⃣ Upload dei Documenti

Carica i tuoi documenti (PDF, immagini, Word, Excel, etc.)

In [ ]:
from google.colab import files
import os

print("📤 Carica i tuoi documenti (PDF, immagini, DOCX, XLSX, PPTX, etc.)")
uploaded = files.upload()

# Salva i nomi dei file caricati
uploaded_files = list(uploaded.keys())
print(f"\n✅ File caricati: {uploaded_files}")

## 4️⃣ Test GPU Mode - Singolo Documento

In [ ]:
from docstrange import DocumentExtractor
import time

# Inizializza con GPU mode
extractor = DocumentExtractor(gpu=True)

# Processa il primo file caricato
if uploaded_files:
    file_path = uploaded_files[0]
    print(f"🔄 Processando: {file_path}...\n")
    
    start_time = time.time()
    result = extractor.extract(file_path)
    elapsed = time.time() - start_time
    
    print(f"\n⚡ Tempo di processamento: {elapsed:.2f}s")
    print(f"📊 Pagine processate: {len(result.pages) if hasattr(result, 'pages') else 'N/A'}")
    print(f"\n" + "="*80)
    print("📝 Risultato (Markdown):")
    print("="*80)
    print(result.to_markdown()[:2000] + "..." if len(result.to_markdown()) > 2000 else result.to_markdown())
else:
    print("⚠️ Nessun file caricato!")

## 5️⃣ Estrazione Dati Strutturati con JSON Schema

In [ ]:
import json

# Esempio: Estrai dati strutturati da una fattura o documento
if uploaded_files:
    # Definisci lo schema JSON per l'estrazione
    schema = {
        "document_type": "string",
        "date": "string",
        "total_amount": "number",
        "items": [
            {
                "description": "string",
                "quantity": "number",
                "price": "number"
            }
        ]
    }
    
    # Estrai dati strutturati
    structured_data = result.extract_data(json_schema=schema)
    
    print("📋 Dati Strutturati Estratti:")
    print("="*80)
    print(json.dumps(structured_data, indent=2, ensure_ascii=False))
else:
    print("⚠️ Carica prima un documento nella cella precedente!")

## 6️⃣ Estrazione Campi Specifici

In [ ]:
# Estrai solo campi specifici
if uploaded_files:
    fields = result.extract_data(
        specified_fields=[
            "invoice_number",
            "date",
            "total_amount",
            "vendor_name",
            "customer_name"
        ]
    )
    
    print("🎯 Campi Specifici Estratti:")
    print("="*80)
    print(json.dumps(fields, indent=2, ensure_ascii=False))
else:
    print("⚠️ Carica prima un documento!")

## 7️⃣ Esportazione in Vari Formati

In [ ]:
from google.colab import files as colab_files

if uploaded_files and result:
    base_name = os.path.splitext(uploaded_files[0])[0]
    
    # Salva in vari formati
    formats = {
        'markdown': result.to_markdown(),
        'text': result.to_text(),
        'html': result.to_html(),
        'json': json.dumps(result.to_dict(), indent=2, ensure_ascii=False)
    }
    
    for fmt, content in formats.items():
        output_file = f"{base_name}.{fmt}"
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(content)
        print(f"✅ Salvato: {output_file}")
    
    # Download tutti i file
    print("\n📥 Download dei file...")
    for fmt in formats.keys():
        colab_files.download(f"{base_name}.{fmt}")
    
    print("\n✅ Download completati!")
else:
    print("⚠️ Carica e processa prima un documento!")

## 8️⃣ Batch Processing - Multipli Documenti

In [ ]:
import time

if len(uploaded_files) > 1:
    print(f"🔄 Processando {len(uploaded_files)} documenti...\n")
    
    results = []
    total_start = time.time()
    
    for i, file_path in enumerate(uploaded_files, 1):
        print(f"[{i}/{len(uploaded_files)}] Processando: {file_path}")
        start = time.time()
        
        try:
            result = extractor.extract(file_path)
            elapsed = time.time() - start
            results.append({
                'file': file_path,
                'success': True,
                'time': elapsed,
                'result': result
            })
            print(f"  ✅ Completato in {elapsed:.2f}s\n")
        except Exception as e:
            print(f"  ❌ Errore: {e}\n")
            results.append({
                'file': file_path,
                'success': False,
                'error': str(e)
            })
    
    total_elapsed = time.time() - total_start
    successful = sum(1 for r in results if r['success'])
    
    print("="*80)
    print(f"📊 Riepilogo Batch Processing")
    print("="*80)
    print(f"Documenti processati: {successful}/{len(uploaded_files)}")
    print(f"Tempo totale: {total_elapsed:.2f}s")
    print(f"Tempo medio per documento: {total_elapsed/len(uploaded_files):.2f}s")
    
elif len(uploaded_files) == 1:
    print("ℹ️ Carica più documenti per testare il batch processing")
else:
    print("⚠️ Carica dei documenti prima!")

## 9️⃣ Confronto: GPU Mode vs Cloud Mode

In [ ]:
if uploaded_files:
    file_path = uploaded_files[0]
    
    print("⚡ Confronto prestazioni GPU vs Cloud...\n")
    
    # Test GPU Mode
    print("🎮 GPU Mode:")
    extractor_gpu = DocumentExtractor(gpu=True)
    start = time.time()
    result_gpu = extractor_gpu.extract(file_path)
    gpu_time = time.time() - start
    print(f"   Tempo: {gpu_time:.2f}s")
    
    # Test Cloud Mode
    print("\n☁️ Cloud Mode:")
    extractor_cloud = DocumentExtractor(gpu=False)
    start = time.time()
    try:
        result_cloud = extractor_cloud.extract(file_path)
        cloud_time = time.time() - start
        print(f"   Tempo: {cloud_time:.2f}s")
        
        print("\n" + "="*80)
        print("📊 Risultati:")
        print("="*80)
        if gpu_time < cloud_time:
            speedup = cloud_time / gpu_time
            print(f"🚀 GPU Mode è {speedup:.1f}x più veloce!")
        else:
            print(f"☁️ Cloud Mode è più veloce in questo caso")
    except Exception as e:
        print(f"   ⚠️ Errore Cloud Mode: {e}")
        print(f"   (Possibile limite rate limit - GPU Mode funziona comunque!)")
else:
    print("⚠️ Carica prima un documento!")

## 🔟 Test con Documento di Esempio

Se non hai documenti, scarica e testa con un PDF di esempio

In [ ]:
# Scarica un PDF di esempio
!wget -q -O sample_invoice.pdf "https://www.africau.edu/images/default/sample.pdf"

print("✅ Documento di esempio scaricato: sample_invoice.pdf")
print("\n🔄 Processando documento di esempio...\n")

extractor = DocumentExtractor(gpu=True)
result = extractor.extract("sample_invoice.pdf")

print("="*80)
print("📝 Risultato:")
print("="*80)
print(result.to_markdown()[:1500] + "...")
print("\n✅ Test completato con successo!")

---

## 📚 Documentazione e Risorse

- **Repository GitHub**: https://github.com/NanoNets/nanonets.ocr-2
- **API Key** (per Cloud Mode): https://app.nanonets.com/#/keys
- **Supporto**: support@nanonets.com

## 💡 Tips:

1. **GPU gratuita su Colab ha limiti di tempo** (~12 ore/giorno)
2. **Per grandi volumi**: Considera Cloud Mode autenticato (10k docs/mese gratis)
3. **Dati sensibili**: GPU Mode è ideale (tutto locale, nessun upload)
4. **Performance**: GPU Mode è più veloce per batch processing

---

### 🎉 Buon divertimento con DocStrange!